In [0]:
-- Limpiamos la tabla antes de cargar (opcional, pero ayuda a no duplicar)
TRUNCATE TABLE bootcamp_gabriel.silver.properties_silver;

-- Insertamos la data transformada
INSERT INTO bootcamp_gabriel.silver.properties_silver
SELECT 
    id,
    TRY_CAST(precio AS DECIMAL(15,2)) as precio,
    UPPER(TRIM(moneda)) as moneda,
    CAST(TRY_CAST(ambientes AS DECIMAL) AS INT) as ambientes,
    TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_totales,
    TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cubiertos,
    cochera,
    CAST(TRY_CAST(antiguedad AS DECIMAL) AS INT) as antiguedad,
    zona as zona_original,
    zona as partido, 
    ubicacion as region,
    url,
    current_timestamp() as _processing_timestamp,
    'properties_bronze' as _source_table
FROM bootcamp_gabriel.bronze.properties_bronze
WHERE precio IS NOT NULL 
  AND TRY_CAST(precio AS DECIMAL) > 0;

In [0]:
CREATE TABLE IF NOT EXISTS bootcamp_gabriel.silver.properties_silver (
    id STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    ambientes INT,
    metros_totales DECIMAL(10,2),
    metros_cubiertos DECIMAL(10,2),
    cochera STRING,
    antiguedad INT,
    zona_original STRING,
    partido STRING,
    region STRING,
    url STRING,
    _processing_timestamp TIMESTAMP,
    _source_table STRING
) USING DELTA
TBLPROPERTIES ('quality' = 'silver');

In [0]:
-- Primero limpiamos para no duplicar si lo corrés dos veces
TRUNCATE TABLE bootcamp_gabriel.silver.properties_silver;

-- Insertamos la data con las transformaciones (CAST y TRY_CAST)
INSERT INTO bootcamp_gabriel.silver.properties_silver
SELECT 
    id,
    TRY_CAST(precio AS DECIMAL(15,2)),
    UPPER(TRIM(moneda)),
    CAST(TRY_CAST(ambientes AS DECIMAL) AS INT),
    TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)),
    TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)),
    cochera,
    CAST(TRY_CAST(antiguedad AS DECIMAL) AS INT),
    zona, -- zona_original
    zona, -- partido (aquí iría el archivo de subzonas si lo tuvieras)
    ubicacion, -- region
    url,
    current_timestamp(),
    'properties_bronze'
FROM bootcamp_gabriel.bronze.properties_bronze
WHERE precio IS NOT NULL; -- Filtro básico para que entren filas